# Week 72, Day 6: Forge

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w072d6.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
from unittest.mock import Mock

m = Mock(side_effect=[1, KeyError("k"), 3])
results = []
for _ in range(3):
    try:
        results.append(m())
    except KeyError:
        results.append("raised")
print(results, m.call_count)

**Predict before you run:** what will this print?



In [ ]:
import pytest

with open("test_w.py", "w") as f:
    f.write('''
import pytest

@pytest.mark.parametrize("a", [1, 2])
@pytest.mark.parametrize("b", ["x", "y", "z"])
def test_pairs(a, b):
    pass

@pytest.fixture(params=[10, 20])
def n(request):
    return request.param

def test_n(n):
    assert n % 10 == 0
''')

pytest.main(["-q", "-p", "no:cacheprovider", "test_w.py"])

**Predict before you run:** what will this print?



In [ ]:
import sys

def f(x):
    if x > 0:
        return "pos"
    return "non-pos"

lines = set()
def tracer(frame, event, arg):
    if frame.f_code is f.__code__ and event == "line":
        lines.add(frame.f_lineno - f.__code__.co_firstlineno)
    return tracer

sys.settrace(tracer)
f(5); f(5)
sys.settrace(None)
print(sorted(lines))

**Predict before you run:** what will this print?



In [ ]:
import ast
tree = ast.parse("ok = total >= 50 and member")
for node in ast.walk(tree):
    if isinstance(node, (ast.Compare, ast.BoolOp)):
        print(type(node).__name__, ast.unparse(node))

**Predict before you run:** what will this print?



In [ ]:
def shrink(n):
    return [c for c in (0, n // 2, n - 1) if c < n]

n = 100
while True:
    for c in shrink(n):
        if c * c > 300:
            n = c
            break
    else:
        break
print(n)

### Exercise: A spy

Write a decorator `spy(func)`: the wrapper passes every call through to `func` and returns its result, recording each call in the wrapper's `calls` list as `(args, kwargs, result)`; if `func` raises, record `(args, kwargs, exception)` and re-raise. Keep the function's name.

In [ ]:
import functools

def spy(func):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_spy():\n    \"recording without replacing\"\n    @spy\n    def divide(a, b=1):\n        return a / b\n    assert divide(6, b=3) == 2 and divide.__name__ == \"divide\"\n    try:\n        divide(1, 0)\n        assert False\n    except ZeroDivisionError:\n        pass\n    assert divide.calls[0] == ((6,), {\"b\": 3}, 2.0)\n    args, kwargs, exc = divide.calls[1]\n    assert args == (1, 0) and isinstance(exc, ZeroDivisionError) and len(divide.calls) == 2")

### Exercise: A disagreement, found and shrunk

Write `disagreement(fast, slow, seed=0, runs=500)`: generate lists of 0 to 6 integers from 0 to 5 with `random.Random(seed)`, and find the first list on which `fast(xs) != slow(xs)`. Then shrink it: repeatedly try removing each single item (first to last), keeping the first removal on which they still disagree, until none does. Return the shrunk list, or `None` if they always agree.

In [ ]:
import random

def disagreement(fast, slow, seed=0, runs=500):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_disagreement():\n    \"an oracle with shrinking\"\n    def slow_unique(xs):\n        out = []\n        for x in xs:\n            if x not in out:\n                out.append(x)\n        return out\n    def fast_wrong(xs):\n        return sorted(set(xs))\n    found = disagreement(fast_wrong, slow_unique)\n    assert found is not None and len(found) == 2 and found[0] > found[1], found\n    assert disagreement(lambda xs: list(dict.fromkeys(xs)), slow_unique) is None")

### Exercise: Forge: a booking system's test suite

Write a pytest suite for `booking.py` that passes on the correct code and catches **all eleven** broken versions. Use fixtures for the parts (a fixed clock, a mailer double, a repository, and the `Bookings` service built from them) and `parametrize` for families of cases. Work out every expected value from the rules above, not from the code.

In [ ]:
# Run this cell first: it creates the files used below.
with open('booking.py', 'w', encoding='utf-8') as f:
    f.write('from dataclasses import dataclass\nfrom datetime import datetime, timedelta\n\n\nclass BookingError(Exception):\n    pass\n\n\n@dataclass(frozen=True)\nclass Booking:\n    room: str\n    start: datetime\n    end: datetime\n    who: str\n\n\nclass MemoryRepo:\n    def __init__(self):\n        self._bookings = []\n\n    def add(self, booking):\n        self._bookings.append(booking)\n\n    def remove(self, booking):\n        self._bookings.remove(booking)\n\n    def for_room(self, room):\n        return sorted((b for b in self._bookings if b.room == room), key=lambda b: b.start)\n\n\nclass Bookings:\n    def __init__(self, repo, clock, mailer, opening=8, closing=20, max_hours=3):\n        self.repo, self.clock, self.mailer = repo, clock, mailer\n        self.opening, self.closing, self.max_hours = opening, closing, max_hours\n\n    def book(self, room, start, hours, who):\n        if hours <= 0 or hours > self.max_hours:\n            raise BookingError("a booking lasts from 1 to 3 hours")\n        if start < self.clock.now():\n            raise BookingError("that time has passed")\n        end = start + timedelta(hours=hours)\n        if start.hour < self.opening or end > start.replace(hour=self.closing, minute=0):\n            raise BookingError("outside opening hours")\n        for other in self.repo.for_room(room):\n            if start < other.end and other.start < end:\n                raise BookingError("that room is already booked")\n        booking = Booking(room, start, end, who)\n        self.repo.add(booking)\n        self.mailer.send(who, f"Booked {room} at {start:%H:%M}")\n        return booking\n\n    def cancel(self, booking, who):\n        if booking.who != who:\n            raise BookingError("you can only cancel your own bookings")\n        self.repo.remove(booking)\n        self.mailer.send(who, f"Cancelled {booking.room} at {booking.start:%H:%M}")\n\n    def free_hours(self, room, day):\n        taken = self.repo.for_room(room)\n        free = []\n        for hour in range(self.opening, self.closing):\n            slot_start = datetime(day.year, day.month, day.day, hour)\n            slot_end = slot_start + timedelta(hours=1)\n            if not any(slot_start < b.end and b.start < slot_end for b in taken):\n                free.append(hour)\n        return free\n')

In [ ]:
from datetime import datetime, date
from unittest.mock import Mock
import pytest
from booking import Bookings, BookingError, MemoryRepo

NOW = datetime(2026, 11, 10, 9, 0)

class FixedClock:
    def __init__(self, now):
        self._now = now

    def now(self):
        return self._now

@pytest.fixture
def bookings():
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("GOOD = open(\"booking.py\").read()\nBROKEN = {\n    \"touching bookings clash\": GOOD.replace(\"if start < other.end and other.start < end:\", \"if start <= other.end and other.start <= end:\"),\n    \"only checks the start\": GOOD.replace(\"if start < other.end and other.start < end:\", \"if other.start <= start < other.end:\"),\n    \"four hours allowed\": GOOD.replace(\"hours > self.max_hours\", \"hours > self.max_hours + 1\"),\n    \"zero hours allowed\": GOOD.replace(\"if hours <= 0 or\", \"if hours < 0 or\"),\n    \"the past allowed\": GOOD.replace(\"if start < self.clock.now():\", \"if start < self.clock.now() - timedelta(days=1):\"),\n    \"late finish allowed\": GOOD.replace(\"end > start.replace(hour=self.closing, minute=0)\", \"start.hour >= self.closing\"),\n    \"anyone can cancel\": GOOD.replace(\"        if booking.who != who:\\n            raise BookingError(\\\"you can only cancel your own bookings\\\")\\n\", \"\"),\n    \"no booking email\": GOOD.replace('        self.mailer.send(who, f\"Booked {room} at {start:%H:%M}\")\\n', \"\"),\n    \"rooms mixed up\": GOOD.replace(\"return sorted((b for b in self._bookings if b.room == room)\", \"return sorted((b for b in self._bookings)\"),\n    \"free hours ignore bookings\": GOOD.replace(\"            if not any(slot_start < b.end and b.start < slot_end for b in taken):\\n\", \"            if True:\\n\"),\n    \"cancel keeps the booking\": GOOD.replace(\"        self.repo.remove(booking)\\n\", \"\"),\n}\ndef test_catches():\n    \"your suite passes on the correct code and catches all eleven broken versions\"\n    assert uses(\"fixture\") and uses(\"parametrize\"), \"use fixtures and parametrize\"\n    results = against(source, \"booking\", {\"correct\": GOOD, **BROKEN})\n    write_file(\"booking.py\", GOOD)\n    assert results[\"correct\"] == [], f\"these fail on the correct code: {results['correct']}\"\n    missed = [k for k in BROKEN if not results[k]]\n    assert not missed, f\"not caught ({len(missed)} of 11): {missed}\"")

> **What made the suite strong.** Look at which of your tests killed which broken version. Most will have been killed by **boundary** cases: exactly 3 hours, exactly 20:00, bookings that touch. A suite of "typical" cases (a 2-hour booking at 11:00, a clash in the middle of another booking) would have had full line coverage and let half the bugs survive. And notice that you never mocked the repository: a real `MemoryRepo` let you check what was stored by reading it back, while the mailer, whose effect is the behaviour, was the one place a mock earned its keep.

### Exercise: Forge bonus: finding flaky tests

Write two detectors. `flaky(history)`: `history` is a list of runs, each a dict with `"commit"` and `"results"` (a dict of test name to `"pass"` or `"fail"`). A test is **flaky** if it both passed and failed on the **same** commit. Return the sorted list of flaky tests. Then `order_dependent(tests, reset)`: `tests` is a list of test functions (a test fails if it raises `AssertionError`) and `reset()` restores the shared state. Run each test **alone** (calling `reset()` before it), then run them all **in order** after a single `reset()`, and finally in **reverse** order after a single `reset()`. Return a sorted list of `(name, how)` pairs for each test whose result differs from its result alone, where `how` is `"in order"` or `"reversed"` (a test can appear twice).

In [ ]:
def flaky(history):
    ___

def order_dependent(tests, reset):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_flaky():\n    \"same commit, different results\"\n    history = [\n        {\"commit\": \"a1\", \"results\": {\"test_x\": \"pass\", \"test_y\": \"pass\", \"test_z\": \"fail\"}},\n        {\"commit\": \"a1\", \"results\": {\"test_x\": \"fail\", \"test_y\": \"pass\", \"test_z\": \"fail\"}},\n        {\"commit\": \"b2\", \"results\": {\"test_x\": \"pass\", \"test_y\": \"fail\", \"test_z\": \"pass\"}},\n        {\"commit\": \"b2\", \"results\": {\"test_x\": \"pass\", \"test_y\": \"fail\", \"test_w\": \"pass\"}},\n    ]\n    assert flaky(history) == [\"test_x\"] and flaky([]) == []\n\ndef test_order():\n    \"order-dependent tests\"\n    state = {\"items\": [], \"logged_in\": False}\n    def reset():\n        state[\"items\"] = []\n        state[\"logged_in\"] = False\n    def test_add():\n        state[\"items\"].append(1)\n        assert len(state[\"items\"]) == 1\n    def test_login():\n        state[\"logged_in\"] = True\n    def test_dashboard():\n        assert state[\"logged_in\"]\n    def test_empty():\n        assert state[\"items\"] == []\n    found = order_dependent([test_add, test_login, test_dashboard, test_empty], reset)\n    assert found == [(\"test_dashboard\", \"in order\"), (\"test_empty\", \"in order\")], found\n    assert order_dependent([test_empty, test_add], reset) == [(\"test_empty\", \"reversed\")]\n    assert order_dependent([test_login], reset) == []")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## Warm-up

## Mixed review

## Forge: a booking system's test suite

`booking.py` (in the exercise) is a small room-booking service, built the Day 5 way: `Bookings(repo, clock, mailer)` takes its storage, clock and mailer as arguments. Its rules:

- `book(room, start, hours, who)` books `room` from `start` for `hours` whole hours. A booking lasts **1 to 3 hours**; it can't start in the **past** (before `clock.now()`); it must start at or after **8:00** and finish by **20:00** on the same day; and it can't **overlap** another booking of the same room (one ending at 10:00 and another starting at 10:00 don't overlap). Any broken rule raises `BookingError`. A successful booking is stored with `repo.add`, confirmed with `mailer.send(who, "Booked Oak at 10:00")` (room and start time), and returned.
- `cancel(booking, who)` removes a booking, but only for the person who made it (`BookingError` otherwise), and sends `"Cancelled Oak at 10:00"`.
- `free_hours(room, day)` lists the whole hours from 8 to 19 at which an hour-long booking of that room on that day would be free.
- `MemoryRepo` is the in-memory repository, with `add`, `remove` and `for_room(room)` (that room's bookings, sorted by start).

## Forge bonus: finding flaky tests

## Weekly quiz

## Week 72 review

Tick off each one honestly. If one feels shaky, go back and redo two exercises from that day.

- I can write yield fixtures, choose scopes, build factories, share fixtures in conftest.py, and parametrize tests and fixtures. ([Day 1](w072d1.html))
- I can program Mock objects, check their calls, patch the right name, and use autospec and fakes. ([Day 2](w072d2.html))
- I can find properties, write them with strategies and given, explain shrinking, and test against a model. ([Day 3](w072d3.html))
- I can measure line and branch coverage, explain its limits, and use mutants to find missing tests. ([Day 4](w072d4.html))
- I can balance kinds of test, inject dependencies, write contract tests and builders, and remove the causes of flakiness. ([Day 5](w072d5.html))
- I wrote a full suite that killed every planted bug, and detected flaky tests automatically. (Today)

Tomorrow is a **rest day**. Next week is **packaging**: turning a folder of code into a project anyone can install, with a proper layout, `pyproject.toml`, versions, changelogs and dependencies.